Designed for Databricks Community Edition or any all-purpose cluster; this notebook has not been executed in a workspace from this repo.

Plain PySpark + Delta equivalent of `src/` for a workspace **without** Delta Live Tables. The DLT rewrite is `dlt_pipeline/orders_dlt.py` and is a separate artifact.

Domain: synthetic e-commerce **order events** (not a `fct_orders` / `dim_*` star schema). Bronze is append-only; silver dedupes and quarantines bad rows; gold is daily metrics.

On Databricks, `spark` is already created and Delta is on the runtime. The cells below use that session. Paths under `/tmp` work on Community Edition; swap in a DBFS or volume path if you prefer.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_SCHEMA = StructType(
    [
        StructField("event_id", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("amount", StringType(), True),
        StructField("event_ts", StringType(), True),
    ]
)

ROOT = "/tmp/databricks-delta-lake"
BRONZE = f"{ROOT}/bronze/order_events"
SILVER = f"{ROOT}/silver/order_events"
QUARANTINE = f"{ROOT}/silver/quarantine"
GOLD_DAILY = f"{ROOT}/gold/daily_order_metrics"
GOLD_MIX = f"{ROOT}/gold/event_type_mix"

Seeded-style sample events, including poison rows (null/empty `event_id`, negative amount, non-numeric amount).

In [ ]:
events = [
    {"event_id": "evt-0001", "order_id": "202401150001", "customer_id": "1", "event_type": "placed", "amount": "12.50", "event_ts": "2024-01-15T01:00:00Z"},
    {"event_id": "evt-0002", "order_id": "202401150001", "customer_id": "1", "event_type": "paid", "amount": "12.50", "event_ts": "2024-01-15T02:00:00Z"},
    {"event_id": "evt-0003", "order_id": "202401150002", "customer_id": "2", "event_type": "shipped", "amount": "40.00", "event_ts": "2024-01-15T03:00:00Z"},
    {"event_id": "evt-0001", "order_id": "202401150001", "customer_id": "1", "event_type": "placed", "amount": "99.00", "event_ts": "2024-01-16T00:00:00Z"},
    {"event_id": None, "order_id": "9990001", "customer_id": "1", "event_type": "placed", "amount": "10.00", "event_ts": "2024-01-15T12:00:00Z"},
    {"event_id": "evt-poison-neg", "order_id": "9990002", "customer_id": "2", "event_type": "refunded", "amount": "-15.50", "event_ts": "2024-01-15T12:00:00Z"},
    {"event_id": "evt-poison-nan", "order_id": "9990003", "customer_id": "3", "event_type": "paid", "amount": "not-a-number", "event_ts": "2024-01-15T12:00:00Z"},
    {"event_id": "", "order_id": "9990004", "customer_id": "4", "event_type": "placed", "amount": "20.00", "event_ts": "2024-01-15T12:00:00Z"},
]
bronze_df = spark.createDataFrame(events, schema=BRONZE_SCHEMA)
bronze_df.write.format("delta").mode("overwrite").save(BRONZE)
spark.read.format("delta").load(BRONZE).show()

Silver: require `event_id`, numeric `amount >= 0`, dedupe on `event_id` (latest `event_ts`). Bad rows go to `silver_quarantine`.

In [ ]:
bronze = spark.read.format("delta").load(BRONZE)
amount_num = F.col("amount").cast("double")
event_id_ok = F.col("event_id").isNotNull() & (F.length(F.trim(F.col("event_id"))) > 0)
amount_ok = amount_num.isNotNull() & (amount_num >= 0)
is_valid = event_id_ok & amount_ok

quarantine = (
    bronze.filter(~is_valid)
    .withColumn(
        "quarantine_reason",
        F.concat_ws(
            ";",
            F.when(~event_id_ok, F.lit("missing_event_id")),
            F.when(~amount_ok, F.lit("invalid_amount")),
        ),
    )
)
quarantine.write.format("delta").mode("overwrite").save(QUARANTINE)

typed = (
    bronze.filter(is_valid)
    .withColumn("event_id", F.trim(F.col("event_id")))
    .withColumn("amount", amount_num)
    .withColumn("order_id", F.col("order_id").cast("long"))
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("event_ts", F.to_timestamp("event_ts"))
    .withColumn("event_date", F.to_date(F.col("event_ts")))
)
window = Window.partitionBy("event_id").orderBy(F.col("event_ts").desc_nulls_last())
silver = typed.withColumn("_rn", F.row_number().over(window)).filter(F.col("_rn") == 1).drop("_rn")
silver.write.format("delta").mode("overwrite").save(SILVER)
print("silver", spark.read.format("delta").load(SILVER).count())
print("quarantine", spark.read.format("delta").load(QUARANTINE).count())

Gold: daily `order_count` and `sum(amount)`, plus `event_type` mix.

In [ ]:
silver = spark.read.format("delta").load(SILVER)
daily = (
    silver.groupBy("event_date")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("amount").alias("amount_sum"),
        F.count("*").alias("event_count"),
    )
)
mix = (
    silver.groupBy("event_date", "event_type")
    .agg(F.count("*").alias("event_count"), F.sum("amount").alias("amount_sum"))
)
daily.write.format("delta").mode("overwrite").save(GOLD_DAILY)
mix.write.format("delta").mode("overwrite").save(GOLD_MIX)
spark.read.format("delta").load(GOLD_DAILY).show()
spark.read.format("delta").load(GOLD_MIX).show()

MERGE upsert: same `event_id` updates in place; a new id inserts. Then time-travel `VERSION AS OF 0` vs current.

In [ ]:
updates = spark.createDataFrame(
    [
        {"event_id": "evt-0001", "order_id": "202401150001", "customer_id": "1", "event_type": "delivered", "amount": "111.11", "event_ts": "2024-01-20T08:00:00Z"},
        {"event_id": "evt-0100", "order_id": "202401210001", "customer_id": "1", "event_type": "placed", "amount": "8.00", "event_ts": "2024-01-21T01:00:00Z"},
    ],
    schema=BRONZE_SCHEMA,
)
typed_updates = (
    updates.withColumn("amount", F.col("amount").cast("double"))
    .withColumn("order_id", F.col("order_id").cast("long"))
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("event_ts", F.to_timestamp("event_ts"))
    .withColumn("event_date", F.to_date(F.col("event_ts")))
)
(
    DeltaTable.forPath(spark, SILVER)
    .alias("t")
    .merge(typed_updates.alias("s"), "t.event_id = s.event_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
earlier = spark.read.format("delta").option("versionAsOf", 0).load(SILVER).count()
current = spark.read.format("delta").load(SILVER).count()
print(f"VERSION AS OF 0: {earlier} rows")
print(f"current: {current} rows")

## VACUUM

Delta `VACUUM` default retention is **7 days (168 hours)**. Do not vacuum this demo table if you still want `VERSION AS OF` on recent commits. Example (not run here):

```python
# spark.sql(f"VACUUM delta.`{SILVER}` RETAIN 168 HOURS")
```